In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import tslearn

# import redpandda
# from redpandda import *
# import geostas

# import visualizations
# import comodo
import hdbscan
import time

from SHiP import SHiP
from SHiP.ultrametric_tree import UltrametricTreeType as UTreeType, AVAILABLE_ULTRAMETRIC_TREE_TYPES
from SHiP.partitioning import PartitioningMethod as PMethod, AVAILABLE_PARTITIONING_METHODS
# from redpandda import preprocessing, preprocess_protein_trajectory
import redpandda_general
from clustering_functions import clustering_workflow
from compare_clusterings import *
import clustering_functions
# from timestep_clustering import *
import distance_matrix as dm
import seaborn as sns
import compare_clusterings as cc


## Preprocessing, dataset selection ###

In [ ]:
#### CAKMAK competitor ###


import numpy as np
import pandas as pd
import os
import time
import logging
import json
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
from st_clustering_benchmark_modified import ST_DBSCAN, ST_Agglomerative, ST_KMeans, ST_OPTICS, ST_SpectralClustering, ST_AffinityPropagation, ST_BIRCH, ST_HDBSCAN


# control execution time of functions
import threading

TIMER = 120
PERMUT = 12

class TimeoutError(Exception):
    pass

class InterruptableThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self._func = func
        self._args = args
        self._kwargs = kwargs
        self._result = None

    def run(self):
        self._result = self._func(*self._args, **self._kwargs)

    @property
    def result(self):
        return self._result


class timeout(object):
    def __init__(self, sec):
        self._sec = sec

    def __call__(self, f):
        def wrapped_f(*args, **kwargs):
            it = InterruptableThread(f, *args, **kwargs)
            it.start()
            it.join(self._sec)
            if not it.is_alive():
                return it.result
            raise TimeoutError('execution expired')
        return wrapped_f
    


def make_generator(parameters):
    """Helper function for st_grid_search. Returns a dictionary of all possible parameter combinations."""
    if not parameters:
        yield dict()
    else:
        key_to_iterate = list(parameters.keys())[0]
        next_round_parameters = {p : parameters[p]
                    for p in parameters if p != key_to_iterate}
        for val in parameters[key_to_iterate]:
            for pars in make_generator(next_round_parameters):
                temp_res = pars
                temp_res[key_to_iterate] = val
                yield temp_res
                
def st_silhouette_score(X, labels, eps1=0.05, eps2=10, metric='euclidean'):
    """Helper function for st_grid_search. Hyperparameter combinations are evaluated with the Silhouette score."""
    n, m = X.shape
    time_dist = pdist(X[:, 0].reshape(n, 1), metric=metric)
    euc_dist = pdist(X[:, 1:], metric=metric)

    # filter the euc_dist matrix using the time_dist
    dist = np.where(time_dist <= eps2, euc_dist, 2 * eps1)

    return silhouette_score(squareform(dist), labels, metric='precomputed')

@timeout(TIMER*PERMUT)
def st_grid_search(estimator, split, X, param_dict, metric, y=None, frame_size=None, frame_overlap=None):
    """
    Grid Search of hyperparameters for spatial-temporal clustering algorithms
    
    Parameters
    ----------
    estimator: class
        ST clustering algorithm
    split: boolean
        Flag to indicate whether whole X should be loaded in RAM or processed in smaller chunks.
    X: numpy array
        Data on which grid search is performed
    param_dict: dict
        Dictionary with parameters to be optimized as keys and value range of grid search as value.
    metric: str
        The metric to evaluate the clustering quality
    y: numpy array
        Optional. Some metrics compare predictions with ground truth. Then, labels need to be provided.
    frame_size: int
        Optional. If split is True, indicate how large the chunks should be.
    
    Returns
    -------
    param_opt
        Optimal hyperparameter combination
    """
    param_opt = None
    s_max = 0
    for param in make_generator(param_dict):
        clust = estimator(**param)
        if not split:
            clust.st_fit(X)
        else:
            clust.st_fit_frame_split(X, frame_size, frame_overlap)
            
        if param_opt is None: 
            param_opt = param
        
        # different performance evaluation metrics
        if metric=='silhouette':
            try:
                score = st_silhouette_score(X=X, labels=clust.labels, eps1=param['eps1'] , eps2=param['eps2'], metric='euclidean')
            except (TypeError, ValueError) as e:
                continue
            #print('Silhouette score for parameters {}: {}'.format(param,score))
        elif metric=='ami':
            score = adjusted_mutual_info_score(y,clust.labels)

        # store parameter combination if it outperforms given the metric
        if score > s_max:
            s_max = score
            param_opt = param
    return param_opt

@timeout(TIMER*PERMUT)
def traj_grid_search(estimator, X, param_dict, metric):
    """
    Grid Search of hyperparameters for spatial-temporal clustering algorithms
    
    Parameters
    ----------
    estimator: class
        ST clustering algorithm
    split: boolean
        Flag to indicate whether whole X should be loaded in RAM or processed in smaller chunks.
    X: numpy array
        Data on which grid search is performed
    param_dict: dict
        Dictionary with parameters to be optimized as keys and value range of grid search as value.
    metric: str
        The metric to evaluate the clustering quality
    y: numpy array
        Optional. Some metrics compare predictions with ground truth. Then, labels need to be provided.
    frame_size: int
        Optional. If split is True, indicate how large the chunks should be.
    
    Returns
    -------
    param_opt
        Optimal hyperparameter combination
    """
    param_opt = {'detect_radius':40, 'similarity_threshold':0.5}
    s_max = 0
    for param in make_generator(param_dict):
        clust = estimator(**param)
        clust.st_fit(X)
        
        if param_opt is None: 
            param_opt = param
        
        # different performance evaluation metrics
        if metric=='silhouette':
            try:
                score = st_silhouette_score(X=X, labels=clust.labels, eps1=param['eps1'] , eps2=param['eps2'], metric='euclidean')
            except (TypeError, ValueError) as e:
                continue
            #print('Silhouette score for parameters {}: {}'.format(param,score))
        # elif metric=='ami':
        #     score = adjusted_mutual_info_score(clust.true_labels,clust.labels)
            #print('AMI score for parameters {}: {}'.format(param,score))
            
        # store parameter combination if it outperforms given the metric
        if score > s_max:
            s_max = score
            param_opt = param
    return param_opt


class Test(object):       
    # use this function with st clusterers
    @timeout(TIMER) # set seconds for timeout
    def frame_split_cluster(self, algorithm, data, frame_size, frame_overlap):
        import time
        start_time = time.time()
        algorithm.st_fit_frame_split(data, frame_size, frame_overlap)
        runtime = time.time() - start_time
        ami = adjusted_mutual_info_score(labels, algorithm.labels)
        return ami, runtime
        
    # use this with trajectory clustering
    @timeout(TIMER)
    def traj_cluster(self,algorithm, data):
        import time
        start_time = time.time()
        algorithm.st_fit(data)
        runtime = time.time() - start_time
        ami = adjusted_mutual_info_score(algorithm.true_labels, algorithm.labels)
        return ami, runtime
        
    # use this with dbscan2
    @timeout(TIMER)
    def cluster(self, algorithm, data):
        import time
        start_time = time.time()
        algorithm.st_fit(data)
        runtime = time.time() - start_time
        ami = adjusted_mutual_info_score(labels, algorithm.labels)
        return ami, runtime
    

def distance_matrix_grid_search(estimator, df, split, X, param_dict, metric, y=None, frame_size=None, frame_overlap=None):
    """
    Grid Search of hyperparameters for distance-based clustering algorithms
    
    Parameters
    ----------
    estimator: class
        ST clustering algorithm
    split: boolean
        Flag to indicate whether whole X should be loaded in RAM or processed in smaller chunks.
    X: numpy array
        Data on which grid search is performed
    param_dict: dict
        Dictionary with parameters to be optimized as keys and value range of grid search as value.
    metric: str
        The metric to evaluate the clustering quality
    y: numpy array
        Optional. Some metrics compare predictions with ground truth. Then, labels need to be provided.
    frame_size: int
        Optional. If split is True, indicate how large the chunks should be.
    
    Returns
    -------
    param_opt
        Optimal hyperparameter combination
    """
    param_opt = None
    s_max = 0
    for param in make_generator(param_dict):
        #clust = estimator(**param)
        #if not split:
        #    clust.st_fit(X)
        #else:
        #    clust.st_fit_frame_split(X, frame_size, frame_overlap)

        # in this case we are getting the results somewhat different


        #result = estimator(X, min_cluster_size=2, min_samples=2 ,return_matrix=False, stdev_addition=False)
        result = estimator(X, return_matrix=False, stdev_addition=False, **param)



        clustering =  list(result[0])

            
        if param_opt is None: 
            param_opt = param
        
        # different performance evaluation metrics
        if metric=='silhouette':
            try:
                score = st_silhouette_score(X=X, labels=clustering, eps1=param['eps1'] , eps2=param['eps2'], metric='euclidean')
            except (TypeError, ValueError) as e:
                continue
            #print('Silhouette score for parameters {}: {}'.format(param,score))
        elif metric=='ami':
            #score = adjusted_mutual_info_score(y,clustering)
            score = score_calc(df,clustering)

        # store parameter combination if it outperforms given the metric
        if score > s_max:
            s_max = score
            param_opt = param
    return param_opt

from sklearn.metrics import adjusted_mutual_info_score
import itertools
import copy
from clustering_functions import clustering_workflow

def clustering_workflow_grid_search_ami(traj_array, matrices_to_apply, base_clustering_algo, param_grid, y_true, post_process_noise=False, noise_label=-1):
    """
    Grid search for optimal clustering parameters using AMI as the scoring metric.

    Parameters
    ----------
    traj_array : numpy array
        The trajectory data.
    matrices_to_apply : list of str
        Types of matrices to generate (e.g., "delta", "stddv").
    base_clustering_algo : dict
        Dictionary with keys: 'name', 'method', and 'params'.
        'params' will be modified with values from param_grid.
    param_grid : dict
        Dictionary of hyperparameter names and lists of values to try.
    y_true : numpy array
        Ground truth labels for AMI evaluation.
    post_process_noise : bool
        Whether to assign noise points after clustering.
    noise_label : int
        Label used to denote noise in clustering.

    Returns
    -------
    best_params : dict
        The hyperparameters with the best AMI score.
    best_score : float
        The highest AMI score achieved.
    best_result : dict
        The result dict from clustering_workflow corresponding to the best score.
    """

    keys, values = zip(*param_grid.items())
    all_param_combos = [dict(zip(keys, v)) for v in itertools.product(*values)]

    best_score = float('-inf')
    best_params = None
    best_result = None

    for param_set in all_param_combos:
        clustering_algo = copy.deepcopy(base_clustering_algo)
        clustering_algo["params"].update(param_set)

        results = clustering_workflow(
            traj_array=traj_array,
            matrices_to_apply=matrices_to_apply,
            clusterings_to_apply=[clustering_algo],
            post_process_noise=post_process_noise,
            noise_label=noise_label,
            return_matrices=False
        )

        if not results:
            continue

        result = results[0]
        labels_pred = result["clustering"]

        try:
            score = adjusted_mutual_info_score(y_true, labels_pred)
        except Exception as e:
            print(f"Error with params {param_set}: {e}")
            continue

        if score > best_score:
            best_score = score
            best_params = param_set
            best_result = result

    return best_params, best_score, best_result

#substitutions = {'frame':'t', 'id':'obj_id','cid':'label','x':'x','y':'y'}
substitutions = {'t':'frame', 'obj_id':'id','label':'cid','x':'x','y':'y'}


def format_cluster_df(df, substitutions, add_z=True):
    filtered_df = df[list(substitutions.values())]
    filtered_df = filtered_df.rename(columns={v: k for k, v in substitutions.items()})
    if add_z:
        if 'z' not in df.columns:
            filtered_df['z'] = 0
    return filtered_df


def append_result(row):
    df_row = pd.DataFrame([row], columns=["dataset", "size", "algorithm", "runtime","n_objects"])
    df_row.to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)

def optimize_clusterings_with_grid_search(traj_array, matrices_to_apply, clusterings_to_apply, param_grids, df, post_process_noise=False, noise_label=-1):
    """
    Perform grid search for each clustering algorithm in clusterings_to_apply for each matrix in matrices_to_apply, 
    optimizing only the parameters specified in param_grids, one matrix at a time.
    
    Parameters
    ----------
    traj_array : numpy array
        The trajectory data.
    matrices_to_apply : list of str
        The list of matrices to generate (e.g., "delta", "stddv").
    clusterings_to_apply : list of dicts
        List of dictionaries where each dict represents a clustering algorithm.
    param_grids : list of dicts
        List of parameter grids for each clustering algorithm to be optimized. If a grid is empty, no optimization will occur.
    df : df
        Ground truth labels for AMI evaluation.
    post_process_noise : bool
        Whether to assign noise points after clustering.
    noise_label : int
        Label used to denote noise.

    Returns
    -------
    optimized_parameters : dict
        A dictionary with matrix types as keys and another dictionary as values. This nested dictionary maps clustering algorithms to their optimized parameters for each matrix type.
    """
    optimized_parameters = {}


    # Loop over each matrix type in matrices_to_apply
    for matrix_type in matrices_to_apply:
        print(f"Optimizing clusterings for matrix type: {matrix_type}")
        
        optimized_parameters[matrix_type] = {}

        # Loop over each clustering algorithm
        for i, clustering_algo in enumerate(clusterings_to_apply):

            param_grid = param_grids[i]  # Get the parameter grid for this algorithm

            # If there are no parameters to optimize, skip the grid search
            if not param_grid:
                print(f"No parameters to optimize for {clustering_algo['name']} with matrix {matrix_type}, skipping grid search.")
                optimized_parameters[matrix_type][clustering_algo['name']] = clustering_algo["params"]
                continue

            # Run grid search for the current matrix-clustering combination
            best_params, best_score, best_result = clustering_workflow_grid_search_ami(
                traj_array=traj_array,
                matrices_to_apply=[matrix_type],  # Only this matrix type
                base_clustering_algo=clustering_algo,
                param_grid=param_grid,
                df=df,
                post_process_noise=post_process_noise,
                noise_label=noise_label
            )

            # Store the optimized parameters for the current matrix-clustering combination
            optimized_parameters[matrix_type][clustering_algo["name"]] = best_params


    return optimized_parameters

def update_clusterings_for_matrix(original_clusterings, optimized_params_dict, matrix_key):
                    updated_clusterings = []
                    for clustering in original_clusterings:
                        name = clustering["name"]
                        updated_params = optimized_params_dict[matrix_key].get(name, clustering["params"])
                        updated_clustering = {
                            "name": clustering["name"],
                            "method": clustering["method"],
                            "params": updated_params
                        }
                        updated_clusterings.append(updated_clustering)
                    return updated_clusterings

In [ ]:
def select_files_balanced(
    data_folder,
    key_func,
    selection_per_model,
    description=""
):
    files_by_model = {"calovi": {}, "reynolds": {}, "couzin": {}}

    # ------------------------------------------------------------
    # Group files by model and key
    # ------------------------------------------------------------
    for f in os.listdir(data_folder):
        if not f.endswith(".csv"):
            continue

        match = pattern.match(f)
        if not match:
            continue

        model = match.group(1)
        full_path = os.path.join(data_folder, f)
        key = key_func(full_path, match)

        if key is None:
            continue

        files_by_model[model].setdefault(key, []).append(f)

    selected = []

    # ------------------------------------------------------------
    # Balanced selection per model
    # ------------------------------------------------------------
    for model, key_dict in files_by_model.items():
        keys = sorted(key_dict.keys())

        if not keys:
            print(f" No valid files for model {model} in {description}")
            continue

        # Shuffle files within each key
        for k in keys:
            random.shuffle(key_dict[k])

        model_selected = []
        key_idx = 0

        while len(model_selected) < selection_per_model:
            k = keys[key_idx % len(keys)]

            if key_dict[k]:
                model_selected.append(key_dict[k].pop())

            key_idx += 1

            # Stop if no files remain anywhere
            if all(len(v) == 0 for v in key_dict.values()):
                break

        selected.extend(model_selected)

        print(
            f" {model}: selected {len(model_selected)} files "
            f"from {len(keys)} timestep values"
        )

    print(f"\n Total selected files: {len(selected)} ({description})")
    return selected


csv_files_timepoints = select_files_balanced(
    data_folder,
    key_timepoints,
    selection_per_model,
    "balanced by number of timepoints"
)

save_selection(
    csv_files_timepoints,
    output_file + "_timepoints_balanced.txt"
)


# Running COMET #

### running SHiP experiments ##

In [ ]:
import math 

def run_and_log(name, model):
    try:
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(MAX_RUNTIME)

        start = time.perf_counter()
        model.fit(data_ts)
        runtime = time.perf_counter() - start

        signal.alarm(0)  # cancel alarm

        pred_labels = model.labels_
        ari = adjusted_rand_score(true_labels, pred_labels)
        nmi = normalized_mutual_info_score(true_labels, pred_labels)

        results.append([
            filename, n_timepoints, n_animals,
            name, round(runtime, 4),
            round(ari, 4), round(nmi, 4)
        ])

        print(f" {name}: runtime={runtime:.3f}s, ARI={ari:.3f}, NMI={nmi:.3f}")
        save_now()

    except TimeoutException:
        signal.alarm(0)
        logging.warning(f"{name} timed out on {filename} (> {MAX_RUNTIME}s)")
        print(f"  {name} skipped on {filename} (timeout)")

        # record timeout in results
        results.append([
            filename, n_timepoints, n_animals,
            f"{name}_TIMEOUT",  # or keep name and add a Status col
            MAX_RUNTIME,
            math.nan,
            math.nan,
        ])
        save_now()

    except Exception as e:
        signal.alarm(0)
        logging.error(f"{name} failed on {filename}: {e}")
        print(f"  {name} failed on {filename}: {e}")

        # record failure in results
        results.append([
            filename, n_timepoints, n_animals,
            f"{name}_ERROR",
            math.nan,
            math.nan,
            math.nan,
        ])
        save_now()

In [ ]:
import os
import time
import signal
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from clustering_functions import clustering_workflow
from SHiP.partitioning import PartitioningMethod as PMethod

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
data_folder = "../COMET/full_data/"
output_file = "00results/runtime_analysis/ship_results_timepoints_thresholdElbow.csv"
output_file = "00results/runtime_analysis/cakmak_ablation_COMET.csv"

MAX_RUNTIME = 200  # seconds

# ------------------------------------------------------------
# Load selected datasets
# ------------------------------------------------------------
with open("00results/runtime_analysis/dataset_selection_timepoints_balanced_ablation.txt", "r") as f:
    selected_files = [line.strip() for line in f if line.strip()]

excluded = {
#     "reynolds_300_38.csv",
    "reynolds_900_51.csv",
#     "reynolds_2400_5.csv",
#     "couzin_300_65.csv",
#     "couzin_2700_96.csv",
}

csv_files = [f for f in selected_files if f not in excluded]
print("number of files to process:", len(csv_files))

# ------------------------------------------------------------
# SHiP configuration
# ------------------------------------------------------------
matrices_to_apply = ["delta+1std"]
matrices_to_apply = ["delta+1std", "stddv"]


clusterings_to_apply = [
    {
        "name": "SHiP",
        "method": "ship",
        "params": {
            "partitioning_method": PMethod.ThresholdElbow,
            "hierarchie": 2,
            "tiebreaker_method": "euclidean_distance",
        },
    }
]

# ------------------------------------------------------------
# Resume existing results
# ------------------------------------------------------------
results = []
processed = set()

if os.path.exists(output_file):
    prev = pd.read_csv(output_file)
    results = prev.values.tolist()
    processed = set(prev["Dataset"])
    print(f"Resume mode: already processed {len(processed)} datasets.")
else:
    print("Starting fresh.")

# ------------------------------------------------------------
# Helper: save incrementally
# ------------------------------------------------------------

def save_now():
    df = pd.DataFrame(
        results,
        columns=["Dataset", "Timepoints", "Animals", "Algorithm","Matrix", "Status", "Runtime", "ARI", "NMI"]
    )
    df.to_csv(output_file, index=False)
    print(" Saved progress.")

# ------------------------------------------------------------
# Timeout handling (Unix only)
# ------------------------------------------------------------
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
runtime_analysis = False

for filename in csv_files:
    # filename = "reynolds_900_50.csv"

    if filename in processed:
        print(f" Skipping {filename}, already processed.")
        continue

    print(f"\n Processing {filename} with COMET...")

    try:
        # ---------------------
        # Load dataset
        # ---------------------
        df_orig = pd.read_csv(os.path.join(data_folder, filename))
        df = df_orig.groupby(['frame', 'id'], as_index=False).mean()
        df = format_cluster_df(df_orig, substitutions)

        traj_array, point_array, n_objects, frames_count = \
            redpandda_general.prepare_data_from_df(df, group_by_obj_id=False)

        print("n obj", n_objects, "frames count", frames_count)
        true_labels = df.groupby("obj_id")["label"].first().to_numpy()

        # ---------------------
        # Run SHiP with timeout
        # ---------------------
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(MAX_RUNTIME)

        start = time.perf_counter()
        res = clustering_workflow(
            traj_array,
            matrices_to_apply,
            clusterings_to_apply,
            post_process_noise=True
        )
        runtime = time.perf_counter() - start

        signal.alarm(0)  # cancel alarm

    except TimeoutException:
        signal.alarm(0)
        print(f" COMET skipped on {filename} (timeout > {MAX_RUNTIME}s)")
        status = "TIMEOUT"
        save_now()
        continue

    except Exception as e:
        signal.alarm(0)
        print(f" COMET failed on {filename}: {e}")
        save_now()
        continue

    # ---------------------
    # Collect results
    # ---------------------
    for r in res:
        matrix_type = r["matrix"]
        pred_labels = r["clustering"]

        if not runtime_analysis:
            ari = adjusted_rand_score(true_labels, pred_labels)
            nmi = normalized_mutual_info_score(true_labels, pred_labels)
        else:
            ari = 0
            nmi = 0

        results.append([
            filename,
            frames_count,
            n_objects,
            "COMET",
            matrix_type,
            "COMPLETED",
            round(runtime, 4),
            round(ari, 4),
            round(nmi, 4),
        ])

        save_now()

print("\n All datasets processed safely!")


### COMET ablation ###

In [ ]:
import os
import time
import signal
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from clustering_functions import clustering_workflow
from SHiP.partitioning import PartitioningMethod as PMethod

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
output_file = "00results/runtime_analysis/simulation_ablation_COMET.csv"
MAX_RUNTIME = 200  # seconds
simulation_file="particle_simulation/particle_sim_random_11.csv"


matrix_list = ["delta","delta+1std","stddv"]


clusterings_to_apply = [
    {"name": "SHiP","method": "ship","params": {"partitioning_method": PMethod.ThresholdElbow,"hierarchie": 2,"tiebreaker_method": "euclidean_distance"}}]

results = []
processed = set()
print("FILE",simulation_file)
for m in matrix_list:
    print("current matrix",m)
    matrices_to_apply = [m]
    if os.path.exists(output_file):
        prev = pd.read_csv(output_file)
        results = prev.values.tolist()
        processed = set(prev["Dataset"])
        print(f"Resume mode: already processed {len(processed)} datasets.")
    else:
        print("Starting fresh.")

    def save_now():
        df = pd.DataFrame(
            results,
            columns=["Dataset", "Timepoints", "Trajectories", "Algorithm","Matrix", "Runtime","ARI", "NMI"]
        )
        df.to_csv(output_file, index=False)
        print(" Saved progress.")

    class TimeoutException(Exception):
        pass

    def timeout_handler(signum, frame):
        raise TimeoutException()
    runtime_analysis = False


    df_orig = pd.read_csv(simulation_file)
    df = df_orig.groupby(['frame', 'id'], as_index=False).mean()
    df = format_cluster_df(df_orig, substitutions)

    traj_array, point_array, frames_count, n_objects = redpandda_general.prepare_data_from_df(df, group_by_obj_id=False)
    true_labels = df.groupby("obj_id")["label"].first().to_numpy()

    start = time.perf_counter()
    res = clustering_workflow(
                traj_array,
                matrices_to_apply,
                clusterings_to_apply,
                post_process_noise=True
            )
    runtime = time.perf_counter() - start

    for r in res:
        pred_labels = r["clustering"]
        matrix_type = r["matrix"]

        if not runtime_analysis:
            ari = adjusted_rand_score(true_labels, pred_labels)
            nmi = normalized_mutual_info_score(true_labels, pred_labels)
        else:
            ari = 0
            nmi = 0
    results.append([
                simulation_file,
                frames_count,
                n_objects,
                "COMET",
                matrix_type,
                round(runtime, 4),
                round(ari, 4),
                round(nmi, 4),])

    save_now()
print("\n All datasets processed safely!")

In [ ]:
# simulation cehck
import os
import time
import signal
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from clustering_functions import clustering_workflow
from SHiP.partitioning import PartitioningMethod as PMethod
import redpandda_general

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
simulation_folder = "../spatio-temporal-clustering-benchmark/dataset_files"  # folder containing your CSV simulations
output_file = "00results/runtime_analysis/cakmak_ablation_COMET.csv"
MAX_RUNTIME = 200  # seconds

matrix_list = ["delta+1std", "stddv"]

clusterings_to_apply = [
    {
        "name": "SHiP",
        "method": "ship",
        "params": {
            "partitioning_method": PMethod.ThresholdElbow,
            "hierarchie": 2,
            "tiebreaker_method": "euclidean_distance"
        }
    }
]

# ------------------------------------------------------------
# Prepare results and resume if necessary
# ------------------------------------------------------------
results = []
processed = set()

if os.path.exists(output_file):
    prev = pd.read_csv(output_file)
    results = prev.values.tolist()
    processed = set(prev["Dataset"])
    print(f"Resume mode: already processed {len(processed)} datasets.")
else:
    print("Starting fresh.")

# ------------------------------------------------------------
# Helper function to save progress
# ------------------------------------------------------------
def save_now():
    df = pd.DataFrame(
        results,
        columns=["Dataset", "Timepoints", "Trajectories", "Algorithm","Matrix", "Runtime","ARI", "NMI"]
    )
    df.to_csv(output_file, index=False)
    print("Saved progress.")

# ------------------------------------------------------------
# Timeout handling (optional, can be used per simulation)
# ------------------------------------------------------------
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

signal.signal(signal.SIGALRM, timeout_handler)

# ------------------------------------------------------------
# Loop over all files in the simulation folder
# ------------------------------------------------------------
sim_files = [os.path.join(simulation_folder, f)
             for f in os.listdir(simulation_folder) if f.endswith(".csv")]
# sim_files.sort()  # optional, ensures numbering order

for simulation_file in sim_files[:100]:
    if simulation_file in processed:
        print(f"Skipping already processed file: {simulation_file}")
        continue

    print(f"\nProcessing file: {simulation_file}")

    try:
        # Load simulation CSV
        df_orig = pd.read_csv(simulation_file)

        # If your format_cluster_df and substitutions exist, uncomment next line:
        df = format_cluster_df(df_orig, substitutions)
        # df = df_orig  # use raw df if not using substitutions

        # Prepare trajectory array for clustering
        traj_array, point_array, frames_count, n_objects = redpandda_general.prepare_data_from_df(
            df, group_by_obj_id=False
        )

        # Ground-truth labels
        if "label" in df.columns:
            true_labels = df.groupby("obj_id")["label"].first().to_numpy()
        else:
            true_labels = None  # or use cluster id column if available

        # Loop over matrices
        for m in matrix_list:
            print(f"  Running matrix: {m}")
            matrices_to_apply = [m]

            start = time.perf_counter()
            res = clustering_workflow(
                traj_array,
                matrices_to_apply,
                clusterings_to_apply,
                post_process_noise=True
            )
            runtime = time.perf_counter() - start

            for r in res:
                pred_labels = r["clustering"]
                matrix_type = r["matrix"]

                if true_labels is not None:
                    ari = adjusted_rand_score(true_labels, pred_labels)
                    nmi = normalized_mutual_info_score(true_labels, pred_labels)
                else:
                    ari = 0
                    nmi = 0

                results.append([
                    simulation_file,
                    frames_count,
                    n_objects,
                    "COMET",
                    matrix_type,
                    round(runtime, 4),
                    round(ari, 4),
                    round(nmi, 4)
                ])

            save_now()

    except Exception as e:
        print(f"ERROR processing {simulation_file}: {e}")
        continue

print("\nAll datasets processed safely!")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load your simulation CSV
df = pd.read_csv(simulation_file)

# Choose a frame to visualize
frame_to_plot = 50
df_frame = df[df['frame'] == frame_to_plot]

# Scatter plot with cluster coloring
plt.figure(figsize=(8,6))
for cid in df_frame['cid'].unique():
    cluster_points = df_frame[df_frame['cid'] == cid]
    plt.scatter(cluster_points['x'], cluster_points['y'], label=f'Cluster {cid}', alpha=0.7)

plt.title(f'Particle positions at frame {frame_to_plot}')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Load simulation data
df = pd.read_csv(simulation_file)

# Setup figure
fig, ax = plt.subplots(figsize=(8,6))

def update(frame):
    ax.clear()
    df_frame = df[df['frame'] == frame]
    for cid in df_frame['cid'].unique():
        cluster_points = df_frame[df_frame['cid'] == cid]
        ax.scatter(cluster_points['x'], cluster_points['y'], label=f'Cluster {cid}', alpha=0.7)
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    ax.set_title(f'Frame {frame}')
    ax.legend()
    ax.grid(True)

# Create animation
ani = animation.FuncAnimation(fig, update, frames=df['frame'].max()+1, interval=50)

# Save to MP4 (requires ffmpeg installed)
ani.save('particle_simulations/particle_simulation.gif', writer='ffmpeg', fps=20)

print("Animation saved to particle_simulation.mp4")


# Competitors (runtime analysis) #

In [ ]:
# up to date (feb 26) — with 200s timeout handling

import os
import time
import signal
import numpy as np
import pandas as pd
from tslearn.clustering import TimeSeriesKMeans, KShape
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import logging

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
data_folder = "../COMET/full_data/"
output_file = "00results/runtime_analysis/tslearn_competitors.csv"
MAX_RUNTIME = 200  # seconds

# ------------------------------------------------------------
# Load selected datasets
# ------------------------------------------------------------
with open("00results/runtime_analysis/dataset_selection_timepoints_balanced.txt", "r") as f:
    selected_files = [line.strip() for line in f if line.strip()]

excluded = {
#     "calovi_1500_42.csv",
#     "reynolds_300_38.csv",
    "reynolds_900_51.csv",
#     "reynolds_2400_5.csv",
#     "couzin_300_65.csv",
#     "couzin_2700_96.csv",
}

csv_files = [f for f in selected_files if f not in excluded]
# ------------------------------------------------------------
# Resume mode
# ------------------------------------------------------------
processed = set()
results = []

if os.path.exists(output_file):
    previous = pd.read_csv(output_file)
    processed = set(previous["Dataset"].unique())
    results = previous.values.tolist()
    print(f"Resuming... already processed: {processed}")

# ------------------------------------------------------------
# Helper: save results safely
# ------------------------------------------------------------
def save_now():
    df = pd.DataFrame(
        results,
        columns=["Dataset", "Timepoints", "Animals", "Algorithm", "Status", "Runtime", "ARI", "NMI"]
    )
    df.to_csv(output_file, index=False)
    print(" Saved progress.")


# ------------------------------------------------------------
# Timeout handling (Unix only)
# ------------------------------------------------------------
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

# ------------------------------------------------------------
# Main Loop
# ------------------------------------------------------------
for filename in csv_files:

    if filename in processed:
        print(f" Skipping {filename}, already processed.")
        continue

    print(f"\n Processing {filename} ...")
    df = pd.read_csv(os.path.join(data_folder, filename))

    # Drop duplicates
    df = df.groupby(['frame', 'id'], as_index=False).mean()

    # Build trajectories
    df_sorted = df.sort_values(["id", "frame"])
    animal_ids = df_sorted["id"].unique()

    traj_list = []
    for aid in animal_ids:
        g = df_sorted[df_sorted["id"] == aid]
        traj_list.append(g[["x", "y"]].values)

    data_ts = np.stack(traj_list, axis=0)
    n_animals, n_timepoints, _ = data_ts.shape

    # Ground truth
    true_labels = df_sorted.groupby("id")["cid"].first().to_numpy()
    n_clusters = len(np.unique(true_labels))
    print("Number of clusters:", n_clusters)

    # --------------------------------------------------------
    # Run competitor algorithms with timeout
    # --------------------------------------------------------
    def run_and_log(name, model):
        try:
            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(MAX_RUNTIME)

            start = time.perf_counter()
            model.fit(data_ts)
            runtime = time.perf_counter() - start

            signal.alarm(0)  # cancel alarm

            pred_labels = model.labels_
            ari = adjusted_rand_score(true_labels, pred_labels)
            nmi = normalized_mutual_info_score(true_labels, pred_labels)

            results.append([
                filename, n_timepoints, n_animals,
                name, round(runtime, 4),
                round(ari, 4), round(nmi, 4)
            ])

            print(f" {name}: runtime={runtime:.3f}s, ARI={ari:.3f}, NMI={nmi:.3f}")
            save_now()

        except TimeoutException:
            signal.alarm(0)
            logging.warning(f"{name} timed out on {filename} (> {MAX_RUNTIME}s)")
            print(f"  {name} skipped on {filename} (timeout)")
            save_now()

        except Exception as e:
            signal.alarm(0)
            logging.error(f"{name} failed on {filename}: {e}")
            save_now()

    # --------------------------------------------------------
    # Algorithms
    # --------------------------------------------------------
    run_and_log("KShape", KShape(n_clusters=n_clusters, random_state=42))
    # run_and_log("TS-KMeans", TimeSeriesKMeans(n_clusters=n_clusters, metric="euclidean", random_state=42))
    # run_and_log("TS-KMeans-DTW", TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", random_state=42))

print("\n Finished all datasets.")
print(" Results stored in:", output_file)
